## Libraries

In [ ]:
import os
import sys
import gc
import argparse
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import json

import rasterio
from rasterio.plot import show
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.windows import from_bounds, bounds as window_bounds

from shapely.geometry import Point, LineString, MultiLineString, box
from shapely.ops import nearest_points

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap, Normalize, ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable
import seaborn as sns

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import contextily as cx

import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as pyo

pio.renderers.default = 'iframe'
pyo.init_notebook_mode(connected=True)

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
mpl.rcParams['figure.dpi'] = 150
sns.set_palette("husl")

## Load SWORD and extract reaches and necessary columns for visualization

### sort out reaches with nlcd only

In [ ]:
# Read the file for urban factors, SWORD reaches, usgs gage heights
sword_urban = gpd.read_parquet(
    "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/usgs_sword_matchups_merritt_v2.parquet"
)

sword_urban = sword_urban.dropna(axis=1, how='all')

In [ ]:
reaches_nlcd = sword_urban.copy()

In [ ]:
reaches_nlcd = reaches_nlcd.drop_duplicates(['reach_id','site_no','nlcd_21_mean', 'nlcd_22_mean','nlcd_23_mean', 'nlcd_24_mean','fctimp_mean'])
reaches_nlcd = reaches_nlcd.to_crs('EPSG:4326')
reaches_nlcd['fctimp_mean'] = reaches_nlcd['fctimp_mean'] / 100

print(f"CRS: {reaches_nlcd.crs}")
print(f"Bounds: {reaches_nlcd.total_bounds}")

In [ ]:
reaches_nlcd.to_parquet('/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_nlcd_merritt_v2.parquet')

### Load reaches

In [ ]:
reaches_nlcd = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_nlcd_merritt_v2.parquet')

In [ ]:
filtered_swot = pd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/swotall_qc_filtered_vD_ver2.parquet')
filtered_swot_id = filtered_swot.reach_id.drop_duplicates().to_list()

In [ ]:
reaches_nlcd = reaches_nlcd[reaches_nlcd.reach_id.isin(filtered_swot_id)]
reaches_nlcd.reach_id.drop_duplicates().shape

In [ ]:
# import json
# mnt_dir = "/nas/cee-hydro/floods/confluence/confluence_version_d_test/version_d_test_mnt"
# reach_ids = reaches_nlcd['reach_id'].astype(str).unique().tolist()

# with open(f'{mnt_dir}/input/reaches_of_interest.json', 'w') as f:
#     json.dump(reach_ids, f)

In [ ]:
# # reach parquet
# print('cycle_id' in df_reach.columns)
# print('pass_id' in df_reach.columns)

# # 또는 time_str로 대체 가능한지
# print(df_reach[['time', 'time_str']].head())

In [ ]:
reaches_nlcd.to_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/all_reaches_vD.parquet')

### ISA stratification sampling (12 reaches)

In [ ]:
def select_12_reaches_no_overlap(gdf, column='fctimp_mean'):
    """
    Select 12 reaches based on percentiles without any overlap
    
    Strategy:
    1. Start from 100% (highest value)
    2. Find closest match
    3. Remove selected from pool
    4. Move to next lower percentile
    5. Repeat
    
    Parameters:
    -----------
    gdf : GeoDataFrame
        Data with reaches
    column : str
        Column to use for stratification
    
    Returns:
    --------
    selected_gdf : GeoDataFrame
        Selected 12 reaches (no overlap)
    """
    
    # Calculate target values
    max_val = gdf[column].max()
    percentages = np.linspace(1, 100, 12)  # [1, 10, 19, 28, ..., 100]
    targets = [max_val * p / 100 for p in percentages]
    
    # Start from highest to lowest (reverse order)
    percentages_reversed = percentages[::-1]  # [100, 91, 82, ...]
    targets_reversed = targets[::-1]
    
    # Track selected indices
    selected_indices = []
    available_pool = gdf.index.tolist()  # All indices available initially
    
    print("Selecting 12 Reaches (No Overlap)")
    
    for p, target in zip(percentages_reversed, targets_reversed):
        # Filter to only available reaches
        available_gdf = gdf.loc[available_pool]
        
        # Find closest match in available pool
        diff = (available_gdf[column] - target).abs()
        closest_idx = diff.idxmin()
        
        # Get the actual value
        actual_val = gdf.loc[closest_idx, column]
        
        print(f"\n{p:.0f}% target: {target:.4f}")
        print(f"  -> Selected: {actual_val:.4f} (diff={abs(actual_val-target):.4f})")
        print(f"     Reach ID: {gdf.loc[closest_idx, 'reach_id']}")
        print(f"     USGS Site: {gdf.loc[closest_idx, 'site_no']}")
        
        # Add to selected list
        selected_indices.append(closest_idx)
        
        # Remove from available pool
        available_pool.remove(closest_idx)
    
    # Get selected reaches (reverse back to ascending order)
    selected_indices_sorted = selected_indices[::-1]  # Back to 1% -> 100%
    selected_gdf = gdf.loc[selected_indices_sorted].copy()
    
    # Add metadata
    selected_gdf['percentile'] = percentages  # 1, 10, 19, ..., 100
    selected_gdf['site_number'] = range(1, 13)
    selected_gdf['target_value'] = targets
    
    print("Selection Complete - No Overlaps!")
    print(f"Selected {len(selected_gdf)} unique reaches")
    print(f"Urban range: {selected_gdf[column].min():.4f} - {selected_gdf[column].max():.4f}")
    
    return selected_gdf


# Execute selection
selected_12 = select_12_reaches_no_overlap(reaches_nlcd, column='fctimp_mean')

In [ ]:
selected_12.to_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.parquet')

### Create interactive plots with nodes, reaches, buffers, and usgs gauges

In [ ]:
selected_12 = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.parquet')

In [ ]:
def snap_point_to_line(point, line):
    """
    Snap a point to the nearest location on a line
    
    Parameters:
    -----------
    point : shapely Point
        Point to snap
    line : shapely LineString or MultiLineString
        Line to snap to
    
    Returns:
    --------
    snapped_point : shapely Point
        Point snapped to line
    """
    # Project point onto line
    distance_along_line = line.project(point)
    snapped_point = line.interpolate(distance_along_line)
    return snapped_point
    
def create_sword_buffer_gage_map_imagery(selected_gdf, sword_shp_path, usgs_csv_path, 
                                          swot_node_path=None, save_path=None):
    """
    Create map with USGS satellite imagery background (no token needed!)
    Now includes SWOT node locations!
    """
    
    print("Loading data...")
    # Load USGS gage locations
    usgs_loc = pd.read_csv(usgs_csv_path, dtype={'gage_id': object})
    usgs_loc = usgs_loc.rename(columns={'gage_id': 'site_no', 
                                        'gage_latitude': 'dec_lat_va',
                                        'gage_longitude': 'dec_long_va',
                                        'gage_name': 'station_nm'})
    
    # Load SWORD reaches
    sword_reaches = gpd.read_file(sword_shp_path).to_crs(epsg=4326)
    selected_reach_ids = selected_gdf['reach_id'].tolist()
    sword_selected = sword_reaches[sword_reaches['reach_id'].isin(selected_reach_ids)].copy()
    
    # Load SWOT node data if provided
    nodes_df = None
    if swot_node_path:
        print("Loading SWOT node data...")
        df_swot = pd.read_parquet(swot_node_path)
        # Get unique nodes for selected reaches
        nodes_df = df_swot[df_swot['reach_id'].isin(selected_reach_ids)].copy()
        # Get unique node locations (assuming columns like node_id, latitude, longitude or p_lat, p_lon)
        # Adjust column names based on your actual data structure
        if 'node_id' in nodes_df.columns:
            nodes_df = nodes_df.drop_duplicates(subset=['node_id'])[['node_id', 'reach_id', 'lat', 'lon']].copy()
            print(f"  Found {len(nodes_df)} unique nodes")
    
    # Prepare buffers
    selected_gdf = selected_gdf.to_crs(epsg=4326).copy().sort_values('site_number').reset_index(drop=True)
    
    # Snap USGS gages
    print("Snapping USGS gages...")
    snapped_gages = []
    for _, row in selected_gdf.iterrows():
        usgs_match = usgs_loc[usgs_loc['site_no'] == row['site_no']]
        if len(usgs_match) > 0:
            gage_point = Point(usgs_match.iloc[0]['dec_long_va'], usgs_match.iloc[0]['dec_lat_va'])
            sword_match = sword_selected[sword_selected['reach_id'] == row['reach_id']]
            if len(sword_match) > 0:
                snapped_point = snap_point_to_line(gage_point, sword_match.iloc[0].geometry)
                snapped_gages.append({
                    'site_no': row['site_no'],
                    'site_number': row['site_number'],
                    'station_nm': usgs_match.iloc[0]['station_nm'],
                    'snapped_lat': snapped_point.y,
                    'snapped_lon': snapped_point.x
                })
    
    snapped_gages_df = pd.DataFrame(snapped_gages)
    
    # Calculate zoom info
    zoom_info = []
    for _, row in selected_gdf.iterrows():
        bounds = row.geometry.bounds
        center_lon = (bounds[0] + bounds[2]) / 2
        center_lat = (bounds[1] + bounds[3]) / 2
        max_range = max(bounds[2] - bounds[0], bounds[3] - bounds[1])
        
        zoom = 12 if max_range < 0.1 else (11 if max_range < 0.2 else (10 if max_range < 0.5 else (9 if max_range < 1 else 8)))
        
        zoom_info.append({
            'site_number': row['site_number'],
            'center_lat': center_lat,
            'center_lon': center_lon,
            'zoom': zoom,
            'reach_id': row['reach_id']
        })
    
    # Create figure
    fig = go.Figure()
    
    # SWORD reaches - cyan color for contrast on satellite imagery
    for _, row in sword_selected.iterrows():
        geom = row.geometry
        if geom.geom_type == 'LineString':
            lons, lats = geom.xy
            fig.add_trace(go.Scattermapbox(
                lon=list(lons), lat=list(lats), mode='lines',
                line=dict(width=4, color='cyan'),
                opacity=0.9, hoverinfo='skip', showlegend=False
            ))
        elif geom.geom_type == 'MultiLineString':
            for line in geom.geoms:
                lons, lats = line.xy
                fig.add_trace(go.Scattermapbox(
                    lon=list(lons), lat=list(lats), mode='lines',
                    line=dict(width=4, color='cyan'),
                    opacity=0.9, hoverinfo='skip', showlegend=False
                ))
    
    # Legend for SWORD reaches
    fig.add_trace(go.Scattermapbox(
        lon=[None], lat=[None], mode='lines',
        line=dict(width=4, color='cyan'),
        name='SWORD Reaches', showlegend=True
    ))
    
    # Buffer polygons
    gdf_json = selected_gdf[['geometry', 'fctimp_mean', 'site_number', 'reach_id', 
                             'site_no', 'percentile', 'nlcd_21_mean', 'nlcd_22_mean',
                             'nlcd_23_mean', 'nlcd_24_mean']].copy()
    gdf_json['id'] = gdf_json.index
    
    centroids = selected_gdf.geometry.centroid
    lons, lats = centroids.x.values, centroids.y.values
    
    hover_text = []
    for idx, row in gdf_json.iterrows():
        text = f"<b>Site #{row['site_number']} ({row['percentile']:.0f}th percentile)</b><br><br>"
        text += f"<b>Reach ID: {row['reach_id']}</b><br>"
        text += f"<b>USGS Site: {row['site_no']}</b><br><br>"
        text += f"<b>Total Impervious: {row['fctimp_mean']:.3f}</b><br><br>"
        text += f"Open Space: {row['nlcd_21_mean']:.3f}<br>"
        text += f"Low Intensity: {row['nlcd_22_mean']:.3f}<br>"
        text += f"Medium Intensity: {row['nlcd_23_mean']:.3f}<br>"
        text += f"High Intensity: {row['nlcd_24_mean']:.3f}<br><br>"
        text += f"Lat: {lats[idx]:.3f}, Lon: {lons[idx]:.3f}"
        hover_text.append(text)
    
    gdf_clean = gdf_json[['geometry', 'fctimp_mean', 'id']].copy()
    geojson_clean = json.loads(gdf_clean.to_json())
    
    fig.add_trace(go.Choroplethmapbox(
        geojson=geojson_clean,
        locations=gdf_clean['id'],
        z=gdf_clean['fctimp_mean'],
        colorscale=[[0, 'rgba(255,255,255,0.3)'], [1, 'rgba(255,0,0,0.7)']],
        zmin=0, zmax=gdf_clean['fctimp_mean'].max(),
        marker=dict(opacity=0.6, line=dict(width=2, color='yellow')),
        colorbar=dict(
            title="Total<br>Impervious",
            tickformat='.3f'
        ),
        hovertemplate='%{customdata}<extra></extra>',
        customdata=hover_text,
        name='Buffer Polygons',
        showlegend=True
    ))
    
    # ============ ADD SWOT NODES ============
    if nodes_df is not None and len(nodes_df) > 0:
        print(f"Adding {len(nodes_df)} SWOT nodes to map...")
        
        node_hover = []
        for _, row in nodes_df.iterrows():
            text = f"<b>SWOT Node</b><br>"
            text += f"Node ID: {row['node_id']}<br>"
            text += f"Reach ID: {row['reach_id']}<br>"
            text += f"Lat: {row['lat']:.4f}, Lon: {row['lon']:.4f}"
            node_hover.append(text)
        
        fig.add_trace(go.Scattermapbox(
            lon=nodes_df['lon'],
            lat=nodes_df['lat'],
            mode='markers',
            marker=dict(size=8, color='lime', symbol='circle', opacity=0.8),
            text=node_hover,
            hovertemplate='%{text}<extra></extra>',
            name='SWOT Nodes',
            showlegend=True
        ))
    
    # USGS gages
    if len(snapped_gages_df) > 0:
        gage_hover = []
        for _, row in snapped_gages_df.iterrows():
            text = f"<b>USGS Gage</b><br>"
            text += f"Site #{row['site_number']}<br>"
            text += f"Site No: {row['site_no']}<br>"
            text += f"Station: {row['station_nm']}"
            gage_hover.append(text)
        
        fig.add_trace(go.Scattermapbox(
            lon=snapped_gages_df['snapped_lon'],
            lat=snapped_gages_df['snapped_lat'],
            mode='markers',
            marker=dict(size=12, color='red', symbol='circle', opacity=1.0),
            text=gage_hover,
            hovertemplate='%{text}<extra></extra>',
            name='USGS Gages',
            showlegend=True
        ))
    
    # Dropdown buttons
    zoom_buttons = [dict(
        label='All Sites (US View)',
        method='relayout',
        args=[{'mapbox.center': {'lat': 39, 'lon': -98}, 'mapbox.zoom': 3.5}]
    )]
    
    for info in zoom_info:
        zoom_buttons.append(dict(
            label=f'Site #{info["site_number"]} - {info["reach_id"]}',
            method='relayout',
            args=[{'mapbox.center': {'lat': info['center_lat'], 'lon': info['center_lon']},
                   'mapbox.zoom': info['zoom']}]
        ))
    
    # KEY PART: USGS Imagery layer!
    fig.update_layout(
        mapbox=dict(
            style="white-bg",
            layers=[{
                "below": 'traces',
                "sourcetype": "raster",
                "sourceattribution": "United States Geological Survey",
                "source": [
                    "https://basemap.nationalmap.gov/arcgis/rest/services/USGSImageryOnly/MapServer/tile/{z}/{y}/{x}"
                ]
            }],
            center={'lat': 39, 'lon': -98},
            zoom=3.5
        ),
        updatemenus=[dict(
            buttons=zoom_buttons,
            direction='down',
            x=0.01, y=0.80,
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='cyan',
            borderwidth=2
        )],
        title='Selected 12 Reaches - USGS Satellite Imagery with SWOT Nodes',
        height=800,
        margin=dict(l=0, r=0, t=50, b=0),
        showlegend=True,
        legend=dict(
            x=0.01, y=0.99,
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=2
        )
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"Saved: {save_path}")
    
    return fig, snapped_gages_df, nodes_df


# Create map with USGS satellite imagery + SWOT nodes
print("Creating Map with USGS Satellite Imagery + SWOT Nodes")

fig_imagery, snapped_gages, nodes_df = create_sword_buffer_gage_map_imagery(
    selected_12,
    sword_shp_path='/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_v1.shp',
    usgs_csv_path="/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv",
    swot_node_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_swot_qc_filtered_vD_node.parquet',
    save_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/map_usgs_imagery_with_nodes.html'
)

print("\nCreated map with USGS satellite imagery + SWOT nodes!")
fig_imagery

### Create Separate plots for paper

In [ ]:
# Fast Version: overlap fixed, white legends, overview border, tight bottom
import gc 
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import ListedColormap
import rasterio
from rasterio.windows import from_bounds
from shapely.geometry import Point
from shapely.ops import nearest_points
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

all_reaches = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/all_reaches_vD.parquet')
selected_12 = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.parquet')


def snap_point_to_line(point, line):
  return nearest_points(point, line)[1]


def get_counterclockwise_offset(site_num, radius=1.8):
  a = np.radians(90 + (site_num - 1) * 50)
  return radius * np.cos(a), radius * np.sin(a)


def lon_formatter(x, pos):
  if x < 0:   return f'{abs(x):.0f}°W'
  elif x > 0: return f'{x:.0f}°E'
  else:       return '0°'   


def lat_formatter(y, pos):
  if y > 0:   return f'{y:.0f}°N'
  elif y < 0: return f'{abs(y):.0f}°S'
  else:       return '0°'   


NLCD_COLORS = {
  11: '#476BA1', 12: '#D1DDF9', 21: '#DDC9C9', 22: '#D89382',
  23: '#ED0000', 24: '#AA0000', 31: '#B2ADA3', 41: '#68AA63',
  42: '#1C6330', 43: '#B5C98E', 52: '#CCBA7C', 71: '#E2E2C1',
  81: '#DDD83E', 82: '#AE7229', 90: '#BBD7ED', 95: '#71A4C1'}


def get_nlcd_cmap():
  colors = ['#000000'] * 256
  for val, color in NLCD_COLORS.items():
      colors[val] = color   
  return ListedColormap(colors)


def get_overview_cmap(mode):
  base = dict(NLCD_COLORS)
  if mode == 'nlcd_gray_urban':
      base[21] = '#E8E8E8'; base[22] = '#CDCDCD'; base[23] = '#9E9E9E'; base[24] = '#6F6F6F'
  cols = [(0, 0, 0, 0)] * 256
  for v, hx in base.items():
      cols[v] = (int(hx[1:3], 16)/255, int(hx[3:5], 16)/255, int(hx[5:7], 16)/255, 1.0)
  return ListedColormap(cols)


def load_nlcd_fast(nlcd_path, view_bounds, downsample=1):
  with rasterio.open(nlcd_path) as src:
      window = from_bounds(view_bounds[0], view_bounds[1], view_bounds[2],
                           view_bounds[3], transform=src.transform)
      if downsample > 1:
          out_shape = (max(1, int(window.height // downsample)),
                       max(1, int(window.width // downsample)))
          nlcd_data = src.read(1, window=window, out_shape=out_shape,
                               resampling=rasterio.enums.Resampling.nearest)
      else:
          nlcd_data = src.read(1, window=window)
      b = rasterio.windows.bounds(window, src.transform)
      extent = [b[0], b[2], b[1], b[3]]
  return nlcd_data, extent


def preload_all_nlcd(nlcd_path, reach_bounds_list, downsample=1, max_workers=48):
  results = {}

  def load_single(args):
      idx, bounds = args
      data, extent = load_nlcd_fast(nlcd_path, bounds, downsample)
      return idx, data, extent

  with ThreadPoolExecutor(max_workers=max_workers) as executor:
      futures = {executor.submit(load_single, (i, b)): i
                 for i, b in enumerate(reach_bounds_list)}
      for future in as_completed(futures):
          idx, data, extent = future.result()
          results[idx] = (data, extent)
          print(f"    Loaded tile {idx+1}/{len(reach_bounds_list)}")
  return results


def create_paper_figure(
  all_reaches_gdf, selected_gdf, sword_shp_path, usgs_csv_path, nlcd_path,
  save_path=None, dpi=300, figsize=(20, 22),
  overview_bg='nlcd_gray_urban', overview_downsample=16,
  # ---- LAYOUT (겹치면 여기) ----
  height_ratios=(2.6, 1.0, 1.2, 1.2, 0.22),  # [overview, nlcd legend, inset1, inset2, data legend]
  hspace=0.30, wspace=0.05,
  overview_legend_loc='lower left',
  overview_border_lw=2.0, overview_border_color='black',
  legend_facecolor='white', legend_edgecolor='gray',
  # ---- FONT SIZES ----
  fs_title=24, fs_legend=24, fs_tick=24, fs_site_num=20,
  fs_nlcd_legend=22, fs_site_title=24, fs_data_legend=18, nlcd_legend_ncol=4,
  # ---- MARKER / POINT SIZES ----
  study_pt_size=60, selected_pt_size=130, gage_marker_size=90,
  legend_ms_study=13, legend_ms_selected=19,
  # ---- LINE WIDTHS ----
  buffer_lw=3.5, sword_lw=3.5, state_lw=0.9,
  # ---- COLORS ----
  study_color='cyan', selected_color='cyan', state_color=None,
):
  start_time = time.time()
  print("Loading vector data...")

  usgs_loc = pd.read_csv(usgs_csv_path, dtype={'gage_id': object})
  usgs_loc = usgs_loc.rename(columns={'gage_id': 'site_no', 'gage_latitude': 'dec_lat_va',
                                      'gage_longitude': 'dec_long_va', 'gage_name': 'station_nm'})
  print(f"Loaded {len(usgs_loc)} USGS gages")

  sword_reaches = gpd.read_file(sword_shp_path).to_crs(epsg=4326)
  sword_selected = sword_reaches[sword_reaches['reach_id'].isin(selected_gdf['reach_id'].tolist())].copy()
  del sword_reaches; gc.collect()
  print(f"Loaded {len(sword_selected)} SWORD reaches")

  print("Loading state boundaries...")
  try:  
      states = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_state_500k.zip")
      continental_states = [
          'WA','OR','CA','NV','ID','MT','WY','UT','AZ','CO','NM','ND','SD','NE','KS','OK','TX',
          'MN','IA','MO','AR','LA','WI','IL','MS','MI','IN','KY','TN','AL','OH','WV','VA','NC',
          'SC','GA','FL','PA','NY','VT','NH','MA','RI','CT','NJ','DE','MD','ME','DC']
      states = states[states['STUSPS'].isin(continental_states)].to_crs(epsg=4326)
  except Exception:
      states = None

  all_reaches_gdf = all_reaches_gdf.to_crs(epsg=4326).copy()
  selected_gdf = selected_gdf.to_crs(epsg=4326).copy()
  if 'site_number' not in selected_gdf.columns:
      if 'Unnamed: 0' in selected_gdf.columns:
          selected_gdf['site_number'] = selected_gdf['Unnamed: 0'] + 1
      else:
          selected_gdf['site_number'] = range(1, len(selected_gdf) + 1)
  selected_gdf = selected_gdf.sort_values('site_number').reset_index(drop=True)

  print("Snapping USGS gages...")
  snapped_gages = []
  for _, row in selected_gdf.iterrows():
      usgs_match = usgs_loc[usgs_loc['site_no'] == row['site_no']]
      if len(usgs_match) > 0:
          gage_point = Point(usgs_match.iloc[0]['dec_long_va'], usgs_match.iloc[0]['dec_lat_va'])
          sword_match = sword_selected[sword_selected['reach_id'] == row['reach_id']]
          if len(sword_match) > 0:
              sp = snap_point_to_line(gage_point, sword_match.iloc[0].geometry)
              snapped_gages.append({'site_no': row['site_no'], 'site_number': row['site_number'],
                                    'snapped_lat': sp.y, 'snapped_lon': sp.x, 'reach_id': row['reach_id']})
  snapped_gages_df = pd.DataFrame(snapped_gages)
  gages_gdf = None
  if len(snapped_gages_df) > 0:
      gages_gdf = gpd.GeoDataFrame(snapped_gages_df,
          geometry=gpd.points_from_xy(snapped_gages_df['snapped_lon'], snapped_gages_df['snapped_lat']),
          crs='EPSG:4326')

  print("Pre-calculating view bounds...")
  reach_bounds_list = []
  for idx, row in selected_gdf.iterrows():
      pb = row.geometry.bounds
      max_range = max(pb[2] - pb[0], pb[3] - pb[1])
      cx = (pb[0] + pb[2]) / 2; cy = (pb[1] + pb[3]) / 2
      half = (max_range / 2) + max_range * 0.15
      reach_bounds_list.append([cx - half, cy - half, cx + half, cy + half])

  print("Loading NLCD tiles in parallel (insets)...")
  nlcd_tiles = preload_all_nlcd(nlcd_path, reach_bounds_list, downsample=1, max_workers=6)
  print(f"NLCD tiles loaded in {time.time() - start_time:.1f}s")

  cmap_nlcd = get_nlcd_cmap()
  fig = plt.figure(figsize=figsize)
  gs = GridSpec(5, 6, figure=fig, height_ratios=list(height_ratios),
                width_ratios=[1, 1, 1, 1, 1, 1], hspace=hspace, wspace=wspace)

  # ---- Row 0: US Overview --------------------------------------------------
  print(f"\n[1] US Overview Map (background={overview_bg})")
  ax_us = fig.add_subplot(gs[0, :])
  ax_us.set_facecolor('white') 
  us_view_bounds = [-125, 24, -66, 50]
  ax_us.set_xlim(us_view_bounds[0], us_view_bounds[2])
  ax_us.set_ylim(us_view_bounds[1], us_view_bounds[3])

  if overview_bg in ('nlcd', 'nlcd_gray_urban'):
      nlcd_ov, ext_ov = load_nlcd_fast(nlcd_path, us_view_bounds, downsample=overview_downsample)
      nlcd_ov_m = np.ma.masked_where(nlcd_ov == 0, nlcd_ov)
      ax_us.imshow(nlcd_ov_m, cmap=get_overview_cmap(overview_bg), extent=ext_ov,
                   aspect='auto', interpolation='nearest', zorder=1, vmin=0, vmax=255)
      del nlcd_ov, nlcd_ov_m; gc.collect()
      auto_state = 'white'
  else: 
      auto_state = '#6e6e6e'

  sc = state_color if state_color is not None else auto_state
  if states is not None:
      states.boundary.plot(ax=ax_us, color=sc, linewidth=state_lw, alpha=0.95, zorder=2)

  all_cx = all_reaches_gdf.geometry.centroid.x.values
  all_cy = all_reaches_gdf.geometry.centroid.y.values
  ax_us.scatter(all_cx, all_cy, c=study_color, s=study_pt_size, alpha=0.95,
                edgecolor='black', linewidth=0.7, marker='o', zorder=4)

  for idx, row in selected_gdf.iterrows():
      c = row.geometry.centroid
      site_num = int(row['site_number'])
      ax_us.scatter(c.x, c.y, c=selected_color, s=selected_pt_size, marker='o',
                    edgecolor='black', linewidth=1.6, zorder=5)
      ox, oy = get_counterclockwise_offset(site_num, radius=1.8)
      lx, ly = c.x + ox, c.y + oy
      ax_us.plot([c.x, lx], [c.y, ly], color='black', linewidth=1.0, zorder=4, alpha=0.8)
      ax_us.annotate(f"{site_num}", xy=(lx, ly), fontsize=fs_site_num, fontweight='bold',
                     color='black', ha='center', va='center',
                     bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                               edgecolor='black', linewidth=0.8, alpha=0.95), zorder=6)

  leg_handles = [ 
      plt.Line2D([0], [0], marker='o', linestyle='None', markerfacecolor=study_color,
                 markeredgecolor='black', markeredgewidth=0.8, markersize=legend_ms_study,
                 label='Study Reaches (n=409)'),
      plt.Line2D([0], [0], marker='o', linestyle='None', markerfacecolor=selected_color,
                 markeredgecolor='black', markeredgewidth=1.6, markersize=legend_ms_selected,
                 label='Selected Sites (n=12)'),
  ]
  ax_us.legend(handles=leg_handles, loc=overview_legend_loc, fontsize=fs_legend,
               frameon=True, fancybox=True, framealpha=1.0,
               edgecolor=legend_edgecolor, facecolor=legend_facecolor)

  ax_us.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
  ax_us.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
  ax_us.tick_params(axis='both', labelsize=fs_tick, width=1.5)
  for lab in ax_us.get_xticklabels() + ax_us.get_yticklabels():
      lab.set_fontweight('bold')
  # 테두리(border)
  for sp in ax_us.spines.values():
      sp.set_visible(True); sp.set_linewidth(overview_border_lw)
      sp.set_edgecolor(overview_border_color); sp.set_zorder(10)
  ax_us.set_title('(a) Study Reaches (n=409)', fontsize=fs_title, fontweight='bold', pad=15)
  ax_us.set_aspect('equal')

  # ---- Row 1: NLCD legend (white bg, 4열=4줄) ------------------------------
  ax_legend_nlcd = fig.add_subplot(gs[1, :]); ax_legend_nlcd.axis('off')
  nlcd_legend_labels = {
      11: 'Open Water', 12: 'Ice/Snow', 21: 'Open Space',
      22: 'Low Developed', 23: 'Medium Developed',
      24: 'High Developed', 31: 'Barren Land', 41: 'Deciduous Forest',
      42: 'Evergreen Forest', 43: 'Mixed Forest', 52: 'Shrubland',
      71: 'Herbaceous', 81: 'Pasture/Hay', 82: 'Cultivated',
      90: 'Woody Wetlands', 95: 'Herba. Wetlands'}
  nlcd_legend_elements = [mpatches.Patch(facecolor=NLCD_COLORS[c], edgecolor='black',
                          linewidth=0.5, label=nlcd_legend_labels[c])
                          for c in [11,12,21,22,23,24,31,41,42,43,52,71,81,82,90,95]]
  ax_legend_nlcd.legend(handles=nlcd_legend_elements, loc='center', ncol=nlcd_legend_ncol,
                        frameon=True, fancybox=True, framealpha=1.0,
                        facecolor=legend_facecolor, edgecolor=legend_edgecolor,
                        prop={'weight': 'bold', 'size': fs_nlcd_legend},
                        columnspacing=0.9, handletextpad=0.5)

  # ---- Rows 2-3: insets (2 x 6) -------------------------------------------
  print("\nPlotting individual reach maps...")
  site_labels = ['b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm']
  for i, (idx, row) in enumerate(selected_gdf.iterrows()):
      ax = fig.add_subplot(gs[(i // 6) + 2, i % 6])
      site_number = int(row['site_number'])
      reach_id = row['reach_id']
      isa_pct = row['fctimp_mean'] * 100
      nlcd_data, extent = nlcd_tiles[i]

      ax.imshow(nlcd_data, cmap=cmap_nlcd, extent=extent, aspect='auto',
                interpolation='nearest', zorder=1, vmin=0, vmax=255)
      ax.set_xlim(reach_bounds_list[i][0], reach_bounds_list[i][2])
      ax.set_ylim(reach_bounds_list[i][1], reach_bounds_list[i][3])

      selected_gdf[selected_gdf['site_number'] == site_number].plot(
          ax=ax, facecolor='none', edgecolor='black', linewidth=buffer_lw, zorder=3)

      sword_match = sword_selected[sword_selected['reach_id'] == reach_id]
      if len(sword_match) > 0: 
          sword_match.plot(ax=ax, color='cyan', linewidth=sword_lw, zorder=4)

      if gages_gdf is not None:
          gage_site = gages_gdf[gages_gdf['site_number'] == site_number]
          if len(gage_site) > 0:
              gage_site.plot(ax=ax, color='black', markersize=gage_marker_size, zorder=6,
                             marker='o', edgecolor='white', linewidth=2)

      ax.set_title(f'({site_labels[i]}) Site {site_number}\n(ISA: {isa_pct:.1f}%)',
                   fontsize=fs_site_title, fontweight='bold', pad=5, linespacing=1.1,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, edgecolor='none'))
      ax.set_aspect('equal'); ax.axis('off')
      print(f"  Site {site_number}")

  # ---- Row 4: data legend (인셋 바로 밑, 빈공간 없게) ----------------------
  ax_legend_data = fig.add_subplot(gs[4, :]); ax_legend_data.axis('off')
  data_legend_elements = [
      mpatches.Patch(facecolor='none', edgecolor='black', linewidth=buffer_lw, label='Reach Buffer Zones'),
      plt.Line2D([0], [0], color='cyan', linewidth=4, label='SWOT Reaches'),
      plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markeredgecolor='white',
                 markersize=10, label='USGS Gages', linestyle='None')]
  ax_legend_data.legend(handles=data_legend_elements, loc='upper center', ncol=3, frameon=True,
                        fancybox=True, framealpha=1.0, facecolor=legend_facecolor,
                        edgecolor=legend_edgecolor, columnspacing=1.5, handletextpad=0.6,
                        prop={'weight': 'bold', 'size': fs_data_legend})

  # tight_layout 사용 안 함 
  del nlcd_tiles; gc.collect()
  print(f"\nFigure created in {time.time() - start_time:.1f}s")

  if save_path:   
      fig.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
      print(f"Saved: {save_path}")
      fig.savefig(save_path.replace('.png', '.pdf'), dpi=dpi, bbox_inches='tight', facecolor='white')
      print(f"Saved: {save_path.replace('.png', '.pdf')}")
  return fig


# Main
fig = create_paper_figure(
  all_reaches,
  selected_12,
  sword_shp_path='/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_v1.shp',
  usgs_csv_path="/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv",
  nlcd_path='/work/pi_kandread_umass_edu/swot-urban/0_data_collection/1_nlcd/Annual_NLCD_LndCov_2024_CU_C1V1_EPSG4326.tif',
  save_path='figures/figure1_paper_figure_study_reaches_nlcd_fast.png',
  dpi=300, 
  figsize=(20, 22),
  overview_bg='nlcd',
  height_ratios=(2.6, 1.0, 1.2, 1.2, 0.22),   # NLCD legend행=1.0 (겹치면 ↑)
  hspace=0.30, wspace=0.05,
  overview_legend_loc='lower left',
  overview_border_lw=2.0, overview_border_color='black',
  legend_facecolor='white', legend_edgecolor='gray',
  # ----- 폰트 ----- 
  fs_title=24, fs_legend=24, fs_tick=24, fs_site_num=16,
  fs_nlcd_legend=22, fs_site_title=24, fs_data_legend=24,
  nlcd_legend_ncol=4,                          # ← 3열은 6줄로 길어져 겹침. 4열=4줄 권장
  # ----- 점 크기 -----
  study_pt_size=60, selected_pt_size=130, gage_marker_size=90,
  legend_ms_study=13, legend_ms_selected=19,
  # ----- 선 굵기 -----
  buffer_lw=3.5, sword_lw=3.5, state_lw=0.9,
  # ----- 색 -----
  study_color='gray', selected_color='cyan', state_color=None,
)
print("\nDone!")  
plt.show()

In [ ]:
# Fast Version with Parallel Loading + Downsampling
import gc
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import ListedColormap
import rasterio
from rasterio.windows import from_bounds
from shapely.geometry import Point
from shapely.ops import nearest_points
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import contextily as cx

# 전체 reaches와 예시 12개 로드
all_reaches = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/all_reaches_vD.parquet')
selected_12 = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.parquet')


def snap_point_to_line(point, line):
    """Snap a point to the nearest location on a line."""
    nearest = nearest_points(point, line)
    return nearest[1]


def get_counterclockwise_offset(site_num, radius=1.8):
    """Get offset for label position in counter-clockwise direction."""
    angle_deg = 90 + (site_num - 1) * 30
    angle_rad = np.radians(angle_deg)
    offset_x = radius * np.cos(angle_rad)
    offset_y = radius * np.sin(angle_rad)
    return offset_x, offset_y


def lon_formatter(x, pos):
    if x < 0:
        return f'{abs(x):.0f}°W'
    elif x > 0:
        return f'{x:.0f}°E'
    else:
        return '0°'


def lat_formatter(y, pos):
    if y > 0:
        return f'{y:.0f}°N'
    elif y < 0:
        return f'{abs(y):.0f}°S'
    else:
        return '0°'


def get_nlcd_cmap():
    """Create NLCD colormap."""
    nlcd_colors = {
        11: '#476BA1', 12: '#D1DDF9', 21: '#DDC9C9', 22: '#D89382', 
        23: '#ED0000', 24: '#AA0000', 31: '#B2ADA3', 41: '#68AA63', 
        42: '#1C6330', 43: '#B5C98E', 52: '#CCBA7C', 71: '#E2E2C1',
        81: '#DDD83E', 82: '#AE7229', 90: '#BBD7ED', 95: '#71A4C1'
    }
    colors = ['#000000'] * 256
    for val, color in nlcd_colors.items():
        colors[val] = color
    return ListedColormap(colors)


def load_nlcd_fast(nlcd_path, view_bounds, downsample=1):
    """
    Fast NLCD loading with optional downsampling.
    Returns (data, extent) tuple.
    """
    with rasterio.open(nlcd_path) as src:
        window = from_bounds(
            view_bounds[0], view_bounds[1], view_bounds[2], view_bounds[3],
            transform=src.transform
        )
        
        if downsample > 1:
            out_shape = (
                max(1, int(window.height // downsample)),
                max(1, int(window.width // downsample))
            )
            nlcd_data = src.read(
                1, window=window, out_shape=out_shape,
                resampling=rasterio.enums.Resampling.nearest
            )
        else:
            nlcd_data = src.read(1, window=window)
        
        actual_bounds = rasterio.windows.bounds(window, src.transform)
        extent = [actual_bounds[0], actual_bounds[2], actual_bounds[1], actual_bounds[3]]
    
    return nlcd_data, extent


def preload_all_nlcd(nlcd_path, reach_bounds_list, downsample=1, max_workers=48):
    """
    Parallel loading of all NLCD tiles.
    """
    results = {}
    
    def load_single(args):
        idx, bounds = args
        data, extent = load_nlcd_fast(nlcd_path, bounds, downsample)
        return idx, data, extent
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(load_single, (i, bounds)): i 
            for i, bounds in enumerate(reach_bounds_list)
        }
        
        for future in as_completed(futures):
            idx, data, extent = future.result()
            results[idx] = (data, extent)
            print(f"    Loaded tile {idx+1}/{len(reach_bounds_list)}")
    
    return results


def create_paper_figure(all_reaches_gdf, selected_gdf, sword_shp_path, usgs_csv_path, nlcd_path,
                        save_path=None, dpi=300):
    """
    Create publication-ready figure with fast parallel NLCD loading.
    """
    
    start_time = time.time()
    
    print("Loading vector data...")
    
    # Load USGS gage locations
    usgs_loc = pd.read_csv(usgs_csv_path, dtype={'gage_id': object})
    usgs_loc = usgs_loc.rename(columns={
        'gage_id': 'site_no', 
        'gage_latitude': 'dec_lat_va',
        'gage_longitude': 'dec_long_va',
        'gage_name': 'station_nm'
    })
    print(f"Loaded {len(usgs_loc)} USGS gages")
    
    # Load SWORD reaches
    sword_reaches = gpd.read_file(sword_shp_path).to_crs(epsg=4326)
    selected_reach_ids = selected_gdf['reach_id'].tolist()
    sword_selected = sword_reaches[sword_reaches['reach_id'].isin(selected_reach_ids)].copy()
    del sword_reaches
    gc.collect()
    print(f"Loaded {len(sword_selected)} SWORD reaches")
    
    # Load US state boundaries
    print("Loading state boundaries...")
    try:
        states = gpd.read_file(
            "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_state_500k.zip"
        )
        continental_states = [
            'WA', 'OR', 'CA', 'NV', 'ID', 'MT', 'WY', 'UT', 'AZ', 'CO', 'NM',
            'ND', 'SD', 'NE', 'KS', 'OK', 'TX', 'MN', 'IA', 'MO', 'AR', 'LA',
            'WI', 'IL', 'MS', 'MI', 'IN', 'KY', 'TN', 'AL', 'OH', 'WV', 'VA',
            'NC', 'SC', 'GA', 'FL', 'PA', 'NY', 'VT', 'NH', 'MA', 'RI', 'CT',
            'NJ', 'DE', 'MD', 'ME', 'DC'
        ]
        states = states[states['STUSPS'].isin(continental_states)].to_crs(epsg=4326)
    except:
        states = None
    
    # Prepare GeoDataFrames
    all_reaches_gdf = all_reaches_gdf.to_crs(epsg=4326).copy()
    selected_gdf = selected_gdf.to_crs(epsg=4326).copy()
    
    if 'site_number' not in selected_gdf.columns:
        if 'Unnamed: 0' in selected_gdf.columns:
            selected_gdf['site_number'] = selected_gdf['Unnamed: 0'] + 1
        else:
            selected_gdf['site_number'] = range(1, len(selected_gdf) + 1)
    
    selected_gdf = selected_gdf.sort_values('site_number').reset_index(drop=True)
    
    # Snap USGS gages
    print("Snapping USGS gages...")
    snapped_gages = []
    for _, row in selected_gdf.iterrows():
        usgs_match = usgs_loc[usgs_loc['site_no'] == row['site_no']]
        if len(usgs_match) > 0:
            gage_point = Point(
                usgs_match.iloc[0]['dec_long_va'], 
                usgs_match.iloc[0]['dec_lat_va']
            )
            sword_match = sword_selected[sword_selected['reach_id'] == row['reach_id']]
            if len(sword_match) > 0:
                snapped_point = snap_point_to_line(gage_point, sword_match.iloc[0].geometry)
                snapped_gages.append({
                    'site_no': row['site_no'],
                    'site_number': row['site_number'],
                    'snapped_lat': snapped_point.y,
                    'snapped_lon': snapped_point.x,
                    'reach_id': row['reach_id']
                })
    
    snapped_gages_df = pd.DataFrame(snapped_gages)
    gages_gdf = None
    if len(snapped_gages_df) > 0:
        gages_gdf = gpd.GeoDataFrame(
            snapped_gages_df,
            geometry=gpd.points_from_xy(
                snapped_gages_df['snapped_lon'], 
                snapped_gages_df['snapped_lat']
            ),
            crs='EPSG:4326'
        )
    
    # Pre-calculate all view bounds for parallel loading
    print("Pre-calculating view bounds...")
    
    reach_bounds_list = []
    for idx, row in selected_gdf.iterrows():
        poly_bounds = row.geometry.bounds
        x_range = poly_bounds[2] - poly_bounds[0]
        y_range = poly_bounds[3] - poly_bounds[1]
        max_range = max(x_range, y_range)
        center_x = (poly_bounds[0] + poly_bounds[2]) / 2
        center_y = (poly_bounds[1] + poly_bounds[3]) / 2
        padding = max_range * 0.15
        half_size = (max_range / 2) + padding
        
        reach_bounds_list.append([
            center_x - half_size,
            center_y - half_size,
            center_x + half_size,
            center_y + half_size
        ])
    
    # Parallel NLCD loading (병렬 로딩)
    print("Loading NLCD tiles in parallel...")
    
    nlcd_tiles = preload_all_nlcd(
        nlcd_path, 
        reach_bounds_list, 
        downsample=1,  # Individual reaches는 원본 해상도
        max_workers=6
    )
    
    print(f"All NLCD tiles loaded in {time.time() - start_time:.1f}s")
    
    # Create figure
    print("Creating figure...")
    
    cmap_nlcd = get_nlcd_cmap()
    
    fig = plt.figure(figsize=(18, 16))
    
    gs = GridSpec(
        5, 6, figure=fig,
        height_ratios=[2.5, 0.15, 1, 1, 0.12],
        width_ratios=[1, 1, 1, 1, 1, 1],
        hspace=0.4, wspace=0.02
    )
    
    # Row 0: US Overview Map
    print("\n[1/13] US Overview Map (satellite + NLCD overlay)")
    ax_us = fig.add_subplot(gs[0, :])
    
    us_view_bounds = [-125, 24, -66, 50]
    
    ax_us.set_xlim(us_view_bounds[0], us_view_bounds[2])
    ax_us.set_ylim(us_view_bounds[1], us_view_bounds[3])
    
    # # 1) 먼저 위성사진 배경
    # cx.add_basemap(
    #     ax_us,
    #     crs='EPSG:4326',
    #     source=cx.providers.Esri.WorldImagery,
    #     zoom=5,
    #     attribution=False
    # )
    
    # 2) NLCD 로드 (다운샘플)
    nlcd_overview, extent_overview = load_nlcd_fast(nlcd_path, us_view_bounds, downsample=16)
    print(f"    Overview NLCD: {nlcd_overview.shape}, {nlcd_overview.nbytes / 1024 / 1024:.1f} MB")
    
    # 3) nodata (0값)를 마스킹
    nlcd_masked = np.ma.masked_where(nlcd_overview == 0, nlcd_overview)
    
    # 4) NLCD colormap (0 = 투명)
    def get_nlcd_cmap_transparent():
        """NLCD colormap with transparent nodata."""
        nlcd_colors = {
            11: '#476BA1', 12: '#D1DDF9', 21: '#DDC9C9', 22: '#D89382', 
            23: '#ED0000', 24: '#AA0000', 31: '#B2ADA3', 41: '#68AA63', 
            42: '#1C6330', 43: '#B5C98E', 52: '#CCBA7C', 71: '#E2E2C1',
            81: '#DDD83E', 82: '#AE7229', 90: '#BBD7ED', 95: '#71A4C1'
        }
        
        colors = [(0, 0, 0, 0)] * 256  # 기본값: 투명
        for val, color in nlcd_colors.items():
            r = int(color[1:3], 16) / 255
            g = int(color[3:5], 16) / 255
            b = int(color[5:7], 16) / 255
            colors[val] = (r, g, b, 1.0)
        
        return ListedColormap(colors)
    
    cmap_nlcd_transparent = get_nlcd_cmap_transparent()
    
    # 5) NLCD 오버레이 (투명 nodata)
    ax_us.imshow(nlcd_masked, cmap=cmap_nlcd_transparent, extent=extent_overview,
                 aspect='auto', interpolation='nearest', zorder=2,
                 vmin=0, vmax=255)
    
    del nlcd_overview, nlcd_masked
    gc.collect()
    
    # 6) Plot state boundaries (흰색)
    if states is not None:
        states.boundary.plot(ax=ax_us, color='white', linewidth=0.8, alpha=0.9, zorder=3)
        
        # for _, state in states.iterrows():
        #     centroid = state.geometry.centroid
        #     if (us_view_bounds[0] < centroid.x < us_view_bounds[2] and
        #         us_view_bounds[1] < centroid.y < us_view_bounds[3]):
        #         ax_us.annotate(
        #             state['STUSPS'],
        #             xy=(centroid.x, centroid.y),
        #             fontsize=10, fontweight='bold',
        #             color='white', alpha=0.95,
        #             ha='center', va='center', zorder=4,
        #             path_effects=[mpl.patheffects.withStroke(linewidth=2.5, foreground='black')]
        #         )
    
    # 7) Plot ALL reaches centroids (파란색) - label 추가
    scatter_all = ax_us.scatter([], [], c='cyan', s=12, alpha=0.9, edgecolor='black', linewidth=0.5, label='Study Reaches (n=409)')
    for idx, row in all_reaches_gdf.iterrows():
        centroid = row.geometry.centroid
        ax_us.scatter(centroid.x, centroid.y, c='cyan', s=12, alpha=0.9, edgecolor='black', linewidth=0.5, zorder=5)
    
    # 8) Highlight selected 12 reaches (흰색 점) - label 추가
    scatter_selected = ax_us.scatter([], [], c='cyan', s=60, edgecolor='black', hatch = 'xxxx',
                                      linewidth=1, label='Selected Sites (n=12)')
    for idx, row in selected_gdf.iterrows():
        centroid = row.geometry.centroid
        site_num = int(row['site_number'])
        
        ax_us.scatter(centroid.x, centroid.y, c='cyan', s=60, edgecolor='black', hatch = 'xxxx',
                                      linewidth=1, zorder=6)
        
        offset_x, offset_y = get_counterclockwise_offset(site_num, radius=1.8)
        label_x = centroid.x + offset_x
        label_y = centroid.y + offset_y
        
        ax_us.plot([centroid.x, label_x], [centroid.y, label_y],
                   color='black', linewidth=0.8, zorder=5, alpha=0.7)
        
        ax_us.annotate(
            f"{site_num}", xy=(label_x, label_y),
            fontsize=11, fontweight='bold', color='black',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                      edgecolor='black', linewidth=0.5, alpha=0.9),
            zorder=7
        )
    
    # # 9) Legend (오른쪽 아래, 흰색 배경)
    # overview_legend_elements = [
    #     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='cyan',markeredgecolor='black', hatch='///',
    #                markersize=8, alpha=0.7, linestyle='None', label='Study Reaches (n=409)'),
    #     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='white',
    #                markeredgecolor='black', markersize=10, linestyle='None', 
    #                label='Selected Sites (n=12)')
    # ]

    scatter_all.set_sizes([60])       # Study Reaches 점 크기
    scatter_selected.set_sizes([100]) # Selected Sites 점 크기
    
    ax_us.legend(
        handles=[scatter_all, scatter_selected],
        # handles=overview_legend_elements,
        loc='lower right',
        fontsize=12,
        frameon=True,
        fancybox=True,
        framealpha=0.95,
        edgecolor='gray',
        facecolor='white'
    )
    
    # 10) 축 포맷
    ax_us.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
    ax_us.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
    ax_us.tick_params(axis='both', labelsize=12, width=1.5)
    
    for label in ax_us.get_xticklabels():
        label.set_fontweight('bold')
    for label in ax_us.get_yticklabels():
        label.set_fontweight('bold')
    
    ax_us.spines['top'].set_visible(1.5)
    ax_us.spines['right'].set_visible(1.5)
    ax_us.spines['bottom'].set_linewidth(1.5)
    ax_us.spines['left'].set_linewidth(1.5)
    
    ax_us.set_title('(a) Study Reaches (n=409)', fontsize=15, fontweight='bold', pad=15)
    ax_us.set_aspect('equal')
    
    # Row 1: NLCD Legend
    ax_legend_nlcd = fig.add_subplot(gs[1, :])
    ax_legend_nlcd.axis('off')
    
    nlcd_legend_colors = {
        11: '#476BA1', 12: '#D1DDF9', 21: '#DDC9C9', 22: '#D89382',
        23: '#ED0000', 24: '#AA0000', 31: '#B2ADA3', 41: '#68AA63',
        42: '#1C6330', 43: '#B5C98E', 52: '#CCBA7C', 71: '#E2E2C1',
        81: '#DBD83D', 82: '#AA7028', 90: '#BAD8EA', 95: '#70A3BA',
    }
    nlcd_legend_labels = {
        11: 'Open Water', 12: 'Perennial Ice/Snow',
        21: 'Developed, Open Space', 22: 'Developed, Low Intensity',
        23: 'Developed, Medium Intensity', 24: 'Developed, High Intensity',
        31: 'Barren Land', 41: 'Deciduous Forest',
        42: 'Evergreen Forest', 43: 'Mixed Forest',
        52: 'Shrub/Scrub', 71: 'Grassland/Herbaceous',
        81: 'Pasture/Hay', 82: 'Cultivated Crops',
        90: 'Woody Wetlands', 95: 'Emergent Herbaceous Wetlands',
    }
    
    nlcd_legend_elements = []
    for cls in [11, 12, 21, 22, 23, 24, 31, 41, 42, 43, 52, 71, 81, 82, 90, 95]:
        nlcd_legend_elements.append(
            mpatches.Patch(facecolor=nlcd_legend_colors[cls], edgecolor='black',
                           linewidth=0.5, label=nlcd_legend_labels[cls])
        )
    
    ax_legend_nlcd.legend(
        handles=nlcd_legend_elements, loc='center', ncol=4,
        frameon=True, fancybox=True, prop={'weight': 'bold', 'size': 11},
        framealpha=0.95, edgecolor='gray', columnspacing=0.8
    )
    
    # Rows 2-3: Individual reach maps (pre-loaded NLCD)
    print("\nPlotting individual reach maps (using pre-loaded tiles)...")
    
    site_labels = ['b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm']
    
    for i, (idx, row) in enumerate(selected_gdf.iterrows()):
        row_idx = (i // 6) + 2
        col_idx = i % 6
        
        ax = fig.add_subplot(gs[row_idx, col_idx])
        
        site_number = int(row['site_number'])
        reach_id = row['reach_id']
        isa_pct = row['fctimp_mean'] * 100
        
        # Get pre-loaded NLCD tile
        nlcd_data, extent = nlcd_tiles[i]
        
        # Plot NLCD (이미 로드됨 - 빠름!)
        ax.imshow(nlcd_data, cmap=cmap_nlcd, extent=extent,
                  aspect='auto', interpolation='nearest', zorder=1,
                  vmin=0, vmax=255)
        
        ax.set_xlim(reach_bounds_list[i][0], reach_bounds_list[i][2])
        ax.set_ylim(reach_bounds_list[i][1], reach_bounds_list[i][3])
        
        # Reach Buffer Zone
        selected_gdf[selected_gdf['site_number'] == site_number].plot(
            ax=ax, facecolor='none', edgecolor='black', linewidth=3.5, zorder=3
        )
        
        # SWOT Reach
        sword_match = sword_selected[sword_selected['reach_id'] == reach_id]
        if len(sword_match) > 0:
            sword_match.plot(ax=ax, color='cyan', linewidth=3.5, zorder=4)
        
        # USGS Gage
        if gages_gdf is not None:
            gage_site = gages_gdf[gages_gdf['site_number'] == site_number]
            if len(gage_site) > 0:
                gage_site.plot(ax=ax, color='black', markersize=80, zorder=6,
                               marker='o', edgecolor='white', linewidth=2)
        
        ax.set_title(f'({site_labels[i]}) Site {site_number} (ISA: {isa_pct:.1f}%)',
                     fontsize=13, fontweight='bold', pad=6,
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                               alpha=0.9, edgecolor='none'))
        ax.set_aspect('equal')
        ax.axis('off')
        
        print(f"  Site {site_number}")
    
    # Row 4: Data Elements Legend
    ax_legend_data = fig.add_subplot(gs[4, 2:4])
    ax_legend_data.axis('off')
    
    data_legend_elements = [
        mpatches.Patch(facecolor='none', edgecolor='black', linewidth=3.5, 
                       label='Reach Buffer Zones'),
        plt.Line2D([0], [0], color='cyan', linewidth=4, label='SWOT Reaches'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='black',
                   markeredgecolor='white', markersize=10, label='USGS Gages', 
                   linestyle='None')
    ]
    
    ax_legend_data.legend(
        handles=data_legend_elements, loc='center', ncol=3,
        frameon=True, fancybox=True,
        framealpha=0.95, edgecolor='gray', columnspacing=1.0, 
        prop={'weight': 'bold', 'size': 13}
    )
    
    plt.tight_layout()
    
    # Clean up NLCD tiles from memory
    del nlcd_tiles
    gc.collect()
    
    total_time = time.time() - start_time
    print(f"\nFigure created in {total_time:.1f}s")
    
    if save_path:
        print("Saving figure...")
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"Saved: {save_path}")
        
        pdf_path = save_path.replace('.png', '.pdf')
        fig.savefig(pdf_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"Saved: {pdf_path}")
    
    return fig


# Main
print("Fast Version with Parallel Loading")

fig = create_paper_figure(
    all_reaches,
    selected_12,
    sword_shp_path='/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_v1.shp',
    usgs_csv_path="/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv",
    nlcd_path='/work/pi_kandread_umass_edu/swot-urban/0_data_collection/1_nlcd/Annual_NLCD_LndCov_2024_CU_C1V1_EPSG4326.tif',
    save_path='figures/figure1_paper_figure_study_reaches_nlcd_fast.png',
    dpi=300
)

print("\nDone!")
plt.show()

### full_version

In [ ]:
# Final Version with NLCD Background + Layout Applied
import gc
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import ListedColormap
import rasterio
from rasterio.windows import from_bounds
from shapely.geometry import Point
from shapely.ops import nearest_points

# 전체 reaches와 예시 12개 로드
all_reaches = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/all_reaches_vD.parquet')
selected_12 = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.parquet')


def snap_point_to_line(point, line):
    """Snap a point to the nearest location on a line."""
    nearest = nearest_points(point, line)
    return nearest[1]


def get_counterclockwise_offset(site_num, radius=1.8):
    """
    Get offset for label position in counter-clockwise direction.
    site_num 1 = 12 o'clock (top), increasing counter-clockwise
    """
    angle_deg = 90 + (site_num - 1) * 30
    angle_rad = np.radians(angle_deg)
    
    offset_x = radius * np.cos(angle_rad)
    offset_y = radius * np.sin(angle_rad)
    
    return offset_x, offset_y


def lon_formatter(x, pos):
    """Format longitude with E/W."""
    if x < 0:
        return f'{abs(x):.0f}°W'
    elif x > 0:
        return f'{x:.0f}°E'
    else:
        return '0°'


def lat_formatter(y, pos):
    """Format latitude with N/S."""
    if y > 0:
        return f'{y:.0f}°N'
    elif y < 0:
        return f'{abs(y):.0f}°S'
    else:
        return '0°'


def get_nlcd_cmap():
    """Create NLCD colormap and legend info."""
    nlcd_colors = {
        11: '#476BA1', 12: '#D1DDF9', 21: '#DDC9C9', 22: '#D89382', 
        23: '#ED0000', 24: '#AA0000', 31: '#B2ADA3', 41: '#68AA63', 
        42: '#1C6330', 43: '#B5C98E', 52: '#CCBA7C', 71: '#E2E2C1',
        81: '#DDD83E', 82: '#AE7229', 90: '#BBD7ED', 95: '#71A4C1'
    }
    
    colors = ['#000000'] * 256
    for val, color in nlcd_colors.items():
        colors[val] = color
    
    cmap = ListedColormap(colors)
    
    return cmap, nlcd_colors


def load_and_plot_nlcd(ax, nlcd_path, view_bounds, cmap, plot_name=""):
    """
    Load NLCD for specific view bounds, plot it, then discard from memory.
    """
    with rasterio.open(nlcd_path) as src:
        window = from_bounds(
            view_bounds[0], view_bounds[1], view_bounds[2], view_bounds[3],
            transform=src.transform
        )
        nlcd_data = src.read(1, window=window)
        actual_bounds = rasterio.windows.bounds(window, src.transform)
        print(f"    {plot_name}: Window {int(window.width)}x{int(window.height)} = "
              f"{int(window.width * window.height):,} pixels, "
              f"{nlcd_data.nbytes / 1024 / 1024:.1f} MB")
    
    extent = [actual_bounds[0], actual_bounds[2], actual_bounds[1], actual_bounds[3]]
    
    ax.imshow(
        nlcd_data, 
        cmap=cmap, 
        extent=extent,
        aspect='auto', 
        interpolation='nearest', 
        zorder=1,
        vmin=0, 
        vmax=255
    )
    
    ax.set_xlim(view_bounds[0], view_bounds[2])
    ax.set_ylim(view_bounds[1], view_bounds[3])
    
    del nlcd_data
    gc.collect()


def create_paper_figure(all_reaches_gdf, selected_gdf, sword_shp_path, usgs_csv_path, nlcd_path,
                        save_path=None, dpi=300):
    """
    Create publication-ready figure with NLCD background.
    Layout applied from previous version.
    """
    
    print("Loading vector data...")
    
    # Load USGS gage locations
    usgs_loc = pd.read_csv(usgs_csv_path, dtype={'gage_id': object})
    usgs_loc = usgs_loc.rename(columns={
        'gage_id': 'site_no', 
        'gage_latitude': 'dec_lat_va',
        'gage_longitude': 'dec_long_va',
        'gage_name': 'station_nm'
    })
    print(f"Loaded {len(usgs_loc)} USGS gages")
    
    # Load SWORD reaches
    sword_reaches = gpd.read_file(sword_shp_path).to_crs(epsg=4326)
    selected_reach_ids = selected_gdf['reach_id'].tolist()
    sword_selected = sword_reaches[sword_reaches['reach_id'].isin(selected_reach_ids)].copy()
    del sword_reaches
    gc.collect()
    print(f"Loaded {len(sword_selected)} SWORD reaches")
    
    # Load US state boundaries
    print("Loading state boundaries...")
    try:
        states = gpd.read_file(
            "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_state_500k.zip"
        )
        continental_states = [
            'WA', 'OR', 'CA', 'NV', 'ID', 'MT', 'WY', 'UT', 'AZ', 'CO', 'NM',
            'ND', 'SD', 'NE', 'KS', 'OK', 'TX', 'MN', 'IA', 'MO', 'AR', 'LA',
            'WI', 'IL', 'MS', 'MI', 'IN', 'KY', 'TN', 'AL', 'OH', 'WV', 'VA',
            'NC', 'SC', 'GA', 'FL', 'PA', 'NY', 'VT', 'NH', 'MA', 'RI', 'CT',
            'NJ', 'DE', 'MD', 'ME', 'DC'
        ]
        states = states[states['STUSPS'].isin(continental_states)].to_crs(epsg=4326)
    except:
        states = None
    
    # Prepare GeoDataFrames
    all_reaches_gdf = all_reaches_gdf.to_crs(epsg=4326).copy()
    selected_gdf = selected_gdf.to_crs(epsg=4326).copy()
    
    total_reaches = 409  # 고정값
    print(f"Total reaches: {total_reaches}")
    
    if 'site_number' not in selected_gdf.columns:
        if 'Unnamed: 0' in selected_gdf.columns:
            selected_gdf['site_number'] = selected_gdf['Unnamed: 0'] + 1
        else:
            selected_gdf['site_number'] = range(1, len(selected_gdf) + 1)
    
    selected_gdf = selected_gdf.sort_values('site_number').reset_index(drop=True)
    
    # Snap USGS gages
    print("Snapping USGS gages...")
    snapped_gages = []
    for _, row in selected_gdf.iterrows():
        usgs_match = usgs_loc[usgs_loc['site_no'] == row['site_no']]
        if len(usgs_match) > 0:
            gage_point = Point(
                usgs_match.iloc[0]['dec_long_va'], 
                usgs_match.iloc[0]['dec_lat_va']
            )
            sword_match = sword_selected[sword_selected['reach_id'] == row['reach_id']]
            if len(sword_match) > 0:
                snapped_point = snap_point_to_line(gage_point, sword_match.iloc[0].geometry)
                snapped_gages.append({
                    'site_no': row['site_no'],
                    'site_number': row['site_number'],
                    'snapped_lat': snapped_point.y,
                    'snapped_lon': snapped_point.x,
                    'reach_id': row['reach_id']
                })
    
    snapped_gages_df = pd.DataFrame(snapped_gages)
    
    gages_gdf = None
    if len(snapped_gages_df) > 0:
        gages_gdf = gpd.GeoDataFrame(
            snapped_gages_df,
            geometry=gpd.points_from_xy(
                snapped_gages_df['snapped_lon'], 
                snapped_gages_df['snapped_lat']
            ),
            crs='EPSG:4326'
        )
    
    # Get NLCD colormap
    cmap_nlcd, nlcd_colors = get_nlcd_cmap()
    
    # Create figure
    print("Creating figure...")
    
    fig = plt.figure(figsize=(18, 16))
    
    gs = GridSpec(
        5, 6, figure=fig,
        height_ratios=[2.5, 0.15, 1, 1, 0.12],  # Overview map 크게
        width_ratios=[1, 1, 1, 1, 1, 1],
        hspace=0.4, wspace=0.02
    )
    
    # Row 0: US Overview Map - CONUS 전체 + NLCD 배경
    print("\n[1/13] US Overview Map")
    ax_us = fig.add_subplot(gs[0, :])
    
    # CONUS 범위
    us_view_bounds = [-125, 24, -66, 50]
    
    # Load and plot NLCD for overview
    load_and_plot_nlcd(ax_us, nlcd_path, us_view_bounds, cmap_nlcd, "Overview")
    
    # Plot state boundaries
    if states is not None:
        states.boundary.plot(ax=ax_us, color='white', linewidth=0.8, alpha=0.9, zorder=2)
        
        for _, state in states.iterrows():
            centroid = state.geometry.centroid
            if (us_view_bounds[0] < centroid.x < us_view_bounds[2] and
                us_view_bounds[1] < centroid.y < us_view_bounds[3]):
                
                ax_us.annotate(
                    state['STUSPS'],
                    xy=(centroid.x, centroid.y),
                    fontsize=10,
                    fontweight='bold',
                    color='white', alpha=0.95,
                    ha='center', va='center', zorder=3,
                    path_effects=[mpl.patheffects.withStroke(linewidth=2.5, foreground='black')]
                )
    
    # Plot ALL reaches centroids - 파란색, 작게, alpha=0.7
    for idx, row in all_reaches_gdf.iterrows():
        centroid = row.geometry.centroid
        ax_us.scatter(
            centroid.x, centroid.y,
            c='blue', s=10, alpha=0.7, zorder=4
        )
    
    # Highlight selected 12 reaches - 흰색 점 + 반시계방향 라벨 (흰색 배경)
    for idx, row in selected_gdf.iterrows():
        centroid = row.geometry.centroid
        site_num = int(row['site_number'])
        
        # 흰색 점
        ax_us.scatter(
            centroid.x, centroid.y,
            c='white', s=40, edgecolor='black', linewidth=1, zorder=5
        )
        
        # 반시계방향으로 라벨 위치 계산
        offset_x, offset_y = get_counterclockwise_offset(site_num, radius=1.8)
        label_x = centroid.x + offset_x
        label_y = centroid.y + offset_y
        
        # 점과 라벨 연결선
        ax_us.plot(
            [centroid.x, label_x], [centroid.y, label_y],
            color='black', linewidth=0.8, zorder=4, alpha=0.7
        )
        
        # 숫자 라벨 (흰색 배경)
        ax_us.annotate(
            f"{site_num}",
            xy=(label_x, label_y),
            fontsize=8, fontweight='bold',
            color='black', ha='center', va='center',
            bbox=dict(
                boxstyle='round,pad=0.2', facecolor='white',
                edgecolor='black', linewidth=0.5, alpha=0.9
            ),
            zorder=6
        )
    
    ax_us.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
    ax_us.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
    ax_us.tick_params(axis='both', labelsize=12, width=1.5)
    
    for label in ax_us.get_xticklabels():
        label.set_fontweight('bold')
    for label in ax_us.get_yticklabels():
        label.set_fontweight('bold')
    
    ax_us.spines['top'].set_visible(False)
    ax_us.spines['right'].set_visible(False)
    ax_us.spines['bottom'].set_linewidth(1.5)
    ax_us.spines['left'].set_linewidth(1.5)
    
    ax_us.set_title('(a) Study Reaches (n=409)', fontsize=15, fontweight='bold', pad=15)
    ax_us.set_aspect('equal')
    
    # Row 1: NLCD Legend - 16개 클래스 (Alaska 제외)
    ax_legend_nlcd = fig.add_subplot(gs[1, :])
    ax_legend_nlcd.axis('off')
    
    nlcd_legend_colors = {
        11: '#476BA1', 12: '#D1DDF9', 21: '#DDC9C9', 22: '#D89382',
        23: '#ED0000', 24: '#AA0000', 31: '#B2ADA3', 41: '#68AA63',
        42: '#1C6330', 43: '#B5C98E', 52: '#CCBA7C', 71: '#E2E2C1',
        81: '#DBD83D', 82: '#AA7028', 90: '#BAD8EA', 95: '#70A3BA',
    }
    
    nlcd_legend_labels = {
        11: 'Open Water', 12: 'Perennial Ice/Snow',
        21: 'Developed, Open Space', 22: 'Developed, Low Intensity',
        23: 'Developed, Medium Intensity', 24: 'Developed, High Intensity',
        31: 'Barren Land', 41: 'Deciduous Forest',
        42: 'Evergreen Forest', 43: 'Mixed Forest',
        52: 'Shrub/Scrub', 71: 'Grassland/Herbaceous',
        81: 'Pasture/Hay', 82: 'Cultivated Crops',
        90: 'Woody Wetlands', 95: 'Emergent Herbaceous Wetlands',
    }
    
    nlcd_legend_elements = []
    for cls in [11, 12, 21, 22, 23, 24, 31, 41, 42, 43, 52, 71, 81, 82, 90, 95]:
        nlcd_legend_elements.append(
            mpatches.Patch(
                facecolor=nlcd_legend_colors[cls], edgecolor='black',
                linewidth=0.5, label=nlcd_legend_labels[cls]
            )
        )
    
    ax_legend_nlcd.legend(
        handles=nlcd_legend_elements, loc='center', ncol=4,
        frameon=True, fancybox=True, prop={'weight': 'bold', 'size': 11},
        framealpha=0.95, edgecolor='gray', columnspacing=0.8
    )
    
    # Rows 2-3: Individual reach maps (selected 12)
    print("\nCreating individual reach maps...")
    
    selected_gdf_sorted = selected_gdf.sort_values('site_number').reset_index(drop=True)
    site_labels = ['b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm']
    
    for i, (idx, row) in enumerate(selected_gdf_sorted.iterrows()):
        row_idx = (i // 6) + 2
        col_idx = i % 6
        
        ax = fig.add_subplot(gs[row_idx, col_idx])
        
        site_number = int(row['site_number'])
        reach_id = row['reach_id']
        isa_pct = row['fctimp_mean'] * 100
        
        print(f"\n[{i+2}/13] Site #{site_number} (ISA: {isa_pct:.1f}%)")
        
        poly_bounds = row.geometry.bounds
        x_range = poly_bounds[2] - poly_bounds[0]
        y_range = poly_bounds[3] - poly_bounds[1]
        max_range = max(x_range, y_range)
        center_x = (poly_bounds[0] + poly_bounds[2]) / 2
        center_y = (poly_bounds[1] + poly_bounds[3]) / 2
        padding = max_range * 0.15
        half_size = (max_range / 2) + padding
        
        reach_view_bounds = [
            center_x - half_size,
            center_y - half_size,
            center_x + half_size,
            center_y + half_size
        ]
        
        # Load and plot NLCD for this reach
        load_and_plot_nlcd(ax, nlcd_path, reach_view_bounds, cmap_nlcd, f"Site {site_number}")
        
        # Reach Buffer Zone - BLACK thick line
        selected_gdf[selected_gdf['site_number'] == site_number].plot(
            ax=ax, facecolor='none', edgecolor='black', linewidth=3.5, zorder=3
        )
        
        # SWOT Reach - cyan
        sword_match = sword_selected[sword_selected['reach_id'] == reach_id]
        if len(sword_match) > 0:
            sword_match.plot(ax=ax, color='cyan', linewidth=3.5, zorder=4)
        
        # USGS Gage - BLACK
        if gages_gdf is not None:
            gage_site = gages_gdf[gages_gdf['site_number'] == site_number]
            if len(gage_site) > 0:
                gage_site.plot(
                    ax=ax, color='black', markersize=80, zorder=6,
                    marker='o', edgecolor='white', linewidth=2
                )
        
        ax.set_title(
            f'({site_labels[i]}) Site {site_number} (ISA: {isa_pct:.1f}%)',
            fontsize=13, fontweight='bold', pad=6,
            bbox=dict(
                boxstyle='round,pad=0.3', facecolor='white',
                alpha=0.9, edgecolor='none'
            )
        )
        ax.set_aspect('equal')
        ax.axis('off')
    
    # Row 4: Data Elements Legend
    ax_legend_data = fig.add_subplot(gs[4, 2:4])
    ax_legend_data.axis('off')
    
    data_legend_elements = [
        mpatches.Patch(facecolor='none', edgecolor='black', linewidth=3.5, 
                       label='Reach Buffer Zone'),
        plt.Line2D([0], [0], color='cyan', linewidth=4, label='SWOT Reach'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='black',
                   markeredgecolor='white', markersize=10, label='USGS Gage', 
                   linestyle='None')
    ]
    
    ax_legend_data.legend(
        handles=data_legend_elements, loc='center', ncol=3,
        frameon=True, fancybox=True,
        framealpha=0.95, edgecolor='gray', columnspacing=1.0, prop={'weight': 'bold', 'size': 13}
    )
    
    plt.tight_layout()
    
    if save_path:
        print("Saving figure...")
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"Saved: {save_path}")
        
        pdf_path = save_path.replace('.png', '.pdf')
        fig.savefig(pdf_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"Saved: {pdf_path}")
    
    return fig


# Main
print("Creating Publication-Ready Figure with NLCD + Layout Applied")

fig = create_paper_figure(
    all_reaches,
    selected_12,
    sword_shp_path='/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_v1.shp',
    usgs_csv_path="/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv",
    nlcd_path='/work/pi_kandread_umass_edu/swot-urban/0_data_collection/1_nlcd/Annual_NLCD_LndCov_2024_CU_C1V1_EPSG4326.tif',
    save_path='figures/figure1_paper_figure_study_reaches_nlcd_final.png',
    dpi=300
)

print("\nDone!")
plt.show()

### Layout check!

In [ ]:
# Layout Test Version - Final v2
import gc
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter
from shapely.geometry import Point
from shapely.ops import nearest_points

# 전체 reaches와 예시 12개 로드
all_reaches = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/all_reaches_vD.parquet')
selected_12 = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.parquet')

def snap_point_to_line(point, line):
    """Snap a point to the nearest location on a line."""
    nearest = nearest_points(point, line)
    return nearest[1]


def get_clockwise_offset(site_num, radius=1.5):
    """
    Get offset for label position in clockwise direction.
    site_num 1 = 12 o'clock (top), increasing clockwise
    """
    # 12시 방향(90°)에서 시작, 시계방향으로 30°씩 감소
    angle_deg = 90 + (site_num - 1) * 30
    angle_rad = np.radians(angle_deg)
    
    offset_x = radius * np.cos(angle_rad)
    offset_y = radius * np.sin(angle_rad)
    
    return offset_x, offset_y


def lon_formatter(x, pos):
    """Format longitude with E/W."""
    if x < 0:
        return f'{abs(x):.0f}°W'
    elif x > 0:
        return f'{x:.0f}°E'
    else:
        return '0°'


def lat_formatter(y, pos):
    """Format latitude with N/S."""
    if y > 0:
        return f'{y:.0f}°N'
    elif y < 0:
        return f'{abs(y):.0f}°S'
    else:
        return '0°'


def create_layout_test(all_reaches_gdf, selected_gdf, sword_shp_path, usgs_csv_path, 
                       save_path=None, dpi=150):
    """
    Create layout figure with feedback applied.
    """
    
    print("Loading vector data...")
    
    # Load USGS gage locations
    usgs_loc = pd.read_csv(usgs_csv_path, dtype={'gage_id': object})
    usgs_loc = usgs_loc.rename(columns={
        'gage_id': 'site_no', 
        'gage_latitude': 'dec_lat_va',
        'gage_longitude': 'dec_long_va',
        'gage_name': 'station_nm'
    })
    print(f"Loaded {len(usgs_loc)} USGS gages")
    
    # Load SWORD reaches
    sword_reaches = gpd.read_file(sword_shp_path).to_crs(epsg=4326)
    selected_reach_ids = selected_gdf['reach_id'].tolist()
    sword_selected = sword_reaches[sword_reaches['reach_id'].isin(selected_reach_ids)].copy()
    del sword_reaches
    gc.collect()
    print(f"Loaded {len(sword_selected)} SWORD reaches")
    
    # Load US state boundaries
    print("Loading state boundaries...")
    try:
        states = gpd.read_file(
            "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_state_500k.zip"
        )
        continental_states = [
            'WA', 'OR', 'CA', 'NV', 'ID', 'MT', 'WY', 'UT', 'AZ', 'CO', 'NM',
            'ND', 'SD', 'NE', 'KS', 'OK', 'TX', 'MN', 'IA', 'MO', 'AR', 'LA',
            'WI', 'IL', 'MS', 'MI', 'IN', 'KY', 'TN', 'AL', 'OH', 'WV', 'VA',
            'NC', 'SC', 'GA', 'FL', 'PA', 'NY', 'VT', 'NH', 'MA', 'RI', 'CT',
            'NJ', 'DE', 'MD', 'ME', 'DC'
        ]
        states = states[states['STUSPS'].isin(continental_states)].to_crs(epsg=4326)
    except:
        states = None
    
    # Prepare GeoDataFrames
    all_reaches_gdf = all_reaches_gdf.to_crs(epsg=4326).copy()
    selected_gdf = selected_gdf.to_crs(epsg=4326).copy()
    
    total_reaches = 409
    print(f"Total reaches: {total_reaches}")
    
    if 'site_number' not in selected_gdf.columns:
        if 'Unnamed: 0' in selected_gdf.columns:
            selected_gdf['site_number'] = selected_gdf['Unnamed: 0'] + 1
        else:
            selected_gdf['site_number'] = range(1, len(selected_gdf) + 1)
    
    selected_gdf = selected_gdf.sort_values('site_number').reset_index(drop=True)
    
    # Snap USGS gages
    print("Snapping USGS gages...")
    snapped_gages = []
    for _, row in selected_gdf.iterrows():
        usgs_match = usgs_loc[usgs_loc['site_no'] == row['site_no']]
        if len(usgs_match) > 0:
            gage_point = Point(
                usgs_match.iloc[0]['dec_long_va'], 
                usgs_match.iloc[0]['dec_lat_va']
            )
            sword_match = sword_selected[sword_selected['reach_id'] == row['reach_id']]
            if len(sword_match) > 0:
                snapped_point = snap_point_to_line(gage_point, sword_match.iloc[0].geometry)
                snapped_gages.append({
                    'site_no': row['site_no'],
                    'site_number': row['site_number'],
                    'snapped_lat': snapped_point.y,
                    'snapped_lon': snapped_point.x,
                    'reach_id': row['reach_id']
                })
    
    snapped_gages_df = pd.DataFrame(snapped_gages)
    
    gages_gdf = None
    if len(snapped_gages_df) > 0:
        gages_gdf = gpd.GeoDataFrame(
            snapped_gages_df,
            geometry=gpd.points_from_xy(
                snapped_gages_df['snapped_lon'], 
                snapped_gages_df['snapped_lat']
            ),
            crs='EPSG:4326'
        )
    
    # Create figure
    print("Creating figure...")
    
    fig = plt.figure(figsize=(18, 16))  # 높이 증가
    
    gs = GridSpec(
        5, 6, figure=fig,
        height_ratios=[2.5, 0.15, 1, 1, 0.12],  # Overview map 크기 증가 (1.8 -> 2.5)
        width_ratios=[1, 1, 1, 1, 1, 1],
        hspace=0.4, wspace=0.02
    )
    
    # Row 0: US Overview Map - 더 크게
    print("\n[1/13] US Overview Map")
    ax_us = fig.add_subplot(gs[0, :])
    
    # CONUS 범위
    us_view_bounds = [-125, 24, -66, 50]
    
    ax_us.set_facecolor('#E8E8E8')
    ax_us.set_xlim(us_view_bounds[0], us_view_bounds[2])
    ax_us.set_ylim(us_view_bounds[1], us_view_bounds[3])
    
    # Plot state boundaries
    if states is not None:
        states.boundary.plot(ax=ax_us, color='gray', linewidth=0.8, alpha=0.9, zorder=2)
        
        for _, state in states.iterrows():
            centroid = state.geometry.centroid
            if (us_view_bounds[0] < centroid.x < us_view_bounds[2] and
                us_view_bounds[1] < centroid.y < us_view_bounds[3]):
                
                ax_us.annotate(
                    state['STUSPS'],
                    xy=(centroid.x, centroid.y),
                    fontsize=10,
                    fontweight='bold',
                    color='dimgray', alpha=0.8,
                    ha='center', va='center', zorder=3
                )
    
    # Plot ALL reaches centroids - 파란색, 작게, alpha=0.7
    for idx, row in all_reaches_gdf.iterrows():
        centroid = row.geometry.centroid
        ax_us.scatter(
            centroid.x, centroid.y,
            c='blue', s=10, alpha=0.7, zorder=4  # 파란색, 작은 사이즈
        )
    
    # Highlight selected 12 reaches - 흰색 점 + 시계방향 라벨 (흰색 배경)
    for idx, row in selected_gdf.iterrows():
        centroid = row.geometry.centroid
        site_num = int(row['site_number'])
        
        # 흰색 점
        ax_us.scatter(
            centroid.x, centroid.y,
            c='white', s=40, edgecolor='black', linewidth=1, zorder=5
        )
        
        # 시계방향으로 라벨 위치 계산
        offset_x, offset_y = get_clockwise_offset(site_num, radius=1.8)
        label_x = centroid.x + offset_x
        label_y = centroid.y + offset_y
        
        # 점과 라벨 연결선
        ax_us.plot(
            [centroid.x, label_x], [centroid.y, label_y],
            color='black', linewidth=0.8, zorder=4, alpha=0.7
        )
        
        # 숫자 라벨 (흰색 배경)
        ax_us.annotate(
            f"{site_num}",
            xy=(label_x, label_y),
            fontsize=8, fontweight='bold',
            color='black', ha='center', va='center',
            bbox=dict(
                boxstyle='round,pad=0.2', facecolor='white',
                edgecolor='black', linewidth=0.5, alpha=0.9
            ),
            zorder=6
        )
    
    ax_us.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
    ax_us.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
    ax_us.tick_params(axis='both', labelsize=12, width=1.5)
    
    for label in ax_us.get_xticklabels():
        label.set_fontweight('bold')
    for label in ax_us.get_yticklabels():
        label.set_fontweight('bold')
    
    ax_us.spines['top'].set_visible(False)
    ax_us.spines['right'].set_visible(False)
    ax_us.spines['bottom'].set_linewidth(1.5)
    ax_us.spines['left'].set_linewidth(1.5)
    
    ax_us.set_title('(a) Study Reaches (n=409)', fontsize=15, fontweight='bold', pad=15)
    ax_us.set_aspect('equal')
    
    # Row 1: NLCD Legend
    ax_legend_nlcd = fig.add_subplot(gs[1, :])
    ax_legend_nlcd.axis('off')
    
    nlcd_colors = {
        11: '#476BA1', 12: '#D1DDF9', 21: '#DDC9C9', 22: '#D89382',
        23: '#ED0000', 24: '#AA0000', 31: '#B2ADA3', 41: '#68AA63',
        42: '#1C6330', 43: '#B5C98E', 52: '#CCBA7C', 71: '#E2E2C1',
        81: '#DBD83D', 82: '#AA7028', 90: '#BAD8EA', 95: '#70A3BA',
    }
    
    nlcd_labels = {
        11: 'Open Water', 12: 'Perennial Ice/Snow',
        21: 'Developed, Open Space', 22: 'Developed, Low Intensity',
        23: 'Developed, Medium Intensity', 24: 'Developed, High Intensity',
        31: 'Barren Land', 41: 'Deciduous Forest',
        42: 'Evergreen Forest', 43: 'Mixed Forest',
        52: 'Shrub/Scrub', 71: 'Grassland/Herbaceous',
        81: 'Pasture/Hay', 82: 'Cultivated Crops',
        90: 'Woody Wetlands', 95: 'Emergent Herbaceous Wetlands',
    }
    
    nlcd_legend_elements = []
    for cls in [11, 12, 21, 22, 23, 24, 31, 41, 42, 43, 52, 71, 81, 82, 90, 95]:
        nlcd_legend_elements.append(
            mpatches.Patch(
                facecolor=nlcd_colors[cls], edgecolor='black',
                linewidth=0.5, label=nlcd_labels[cls]
            )
        )
    
    ax_legend_nlcd.legend(
        handles=nlcd_legend_elements, loc='center', ncol=4,
        frameon=True, fancybox=True, prop={'weight': 'bold', 'size': 11},
        framealpha=0.95, edgecolor='gray', columnspacing=0.8
    )
    
    # Rows 2-3: Individual reach maps (selected 12)
    print("\nCreating individual reach maps...")
    
    selected_gdf_sorted = selected_gdf.sort_values('site_number').reset_index(drop=True)
    site_labels = ['b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm']
    
    for i, (idx, row) in enumerate(selected_gdf_sorted.iterrows()):
        row_idx = (i // 6) + 2
        col_idx = i % 6
        
        ax = fig.add_subplot(gs[row_idx, col_idx])
        
        site_number = int(row['site_number'])
        reach_id = row['reach_id']
        isa_pct = row['fctimp_mean'] * 100
        
        print(f"  Site #{site_number} (ISA: {isa_pct:.1f}%)")
        
        poly_bounds = row.geometry.bounds
        x_range = poly_bounds[2] - poly_bounds[0]
        y_range = poly_bounds[3] - poly_bounds[1]
        max_range = max(x_range, y_range)
        center_x = (poly_bounds[0] + poly_bounds[2]) / 2
        center_y = (poly_bounds[1] + poly_bounds[3]) / 2
        padding = max_range * 0.15
        half_size = (max_range / 2) + padding
        
        reach_view_bounds = [
            center_x - half_size,
            center_y - half_size,
            center_x + half_size,
            center_y + half_size
        ]
        
        ax.set_facecolor('#E8E8E8')
        ax.set_xlim(reach_view_bounds[0], reach_view_bounds[2])
        ax.set_ylim(reach_view_bounds[1], reach_view_bounds[3])
        
        # Reach Buffer Zone - BLACK thick line
        selected_gdf[selected_gdf['site_number'] == site_number].plot(
            ax=ax, facecolor='none', edgecolor='black', linewidth=3.5, zorder=3
        )
        
        # SWOT Reach - cyan
        sword_match = sword_selected[sword_selected['reach_id'] == reach_id]
        if len(sword_match) > 0:
            sword_match.plot(ax=ax, color='cyan', linewidth=3.5, zorder=4)
        
        # USGS Gage - BLACK
        if gages_gdf is not None:
            gage_site = gages_gdf[gages_gdf['site_number'] == site_number]
            if len(gage_site) > 0:
                gage_site.plot(
                    ax=ax, color='black', markersize=80, zorder=6,
                    marker='o', edgecolor='white', linewidth=2
                )
        
        ax.set_title(
            f'({site_labels[i]}) Site {site_number} (ISA: {isa_pct:.1f}%)',
            fontsize=13, fontweight='bold', pad=6,
            bbox=dict(
                boxstyle='round,pad=0.3', facecolor='white',
                alpha=0.9, edgecolor='none'
            )
        )
        ax.set_aspect('equal')
        ax.axis('off')
    
    # Row 4: Data Elements Legend
    ax_legend_data = fig.add_subplot(gs[4, 2:4])
    ax_legend_data.axis('off')
    
    data_legend_elements = [
        mpatches.Patch(facecolor='none', edgecolor='black', linewidth=3.5, 
                       label='Reach Buffer Zone'),
        plt.Line2D([0], [0], color='cyan', linewidth=4, label='SWOT Reach'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='black',
                   markeredgecolor='white', markersize=10, label='USGS Gage', 
                   linestyle='None')
    ]
    
    ax_legend_data.legend(
        handles=data_legend_elements, loc='center', ncol=3,
        frameon=True, fancybox=True,
        framealpha=0.95, edgecolor='gray', columnspacing=1.0, prop={'weight': 'bold', 'size': 13}
    )
    
    plt.tight_layout()
    
    if save_path:
        print("Saving figure...")
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"Saved: {save_path}")
    
    return fig


# Main
print("Layout Test - Final v2")

fig = create_layout_test(
    all_reaches,
    selected_12,
    sword_shp_path='/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_v1.shp',
    usgs_csv_path="/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv",
    save_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/layout_test_final_v2.png',
    dpi=150
)

print("\nDone!")
plt.show()